# Two-Stage Logical Fallacy Detection

A single system that uses **all** the datasets, each for the job it's labeled for:

```
            Stage 1: DETECT                     Stage 2: TYPE
  text ─▶  fallacy?  ── valid ─▶ "no fallacy"
                    └─ fallacy ─▶  which of 13 types?  ─▶  e.g. "ad hominem"
```

- **Stage 1 — Detector** (binary, `contrastive_dataset.csv`): is this argument fallacious or valid?
  The only set with *non-fallacy* negatives.
- **Stage 2 — Typer** (13-class, LOGIC `edu_*`): if it's a fallacy, which kind?
  Trained on `edu` only and tested **cross-domain** on real-world climate claims (`climate_test.csv`).

Each stage benchmarks three models — **TF-IDF+LogReg → from-scratch BiLSTM → fine-tuned
DistilBERT** — reported with **macro-F1**. MAFALDA (span-level, different taxonomy) is left
as a future benchmark.

> Colab-ready. Turn on a GPU (Runtime → Change runtime type → T4 GPU) for the DistilBERT parts.

## 0. Setup

In [ ]:
%pip install -q torch transformers scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
import os, re, glob, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter

warnings.filterwarnings("ignore")
%matplotlib inline

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE)

## 0b. Environment (Colab-ready)
Reads the data from the **local Colab session** (the files you uploaded to the VM, which live
under `/content`) — no Google Drive. Runs locally too (falls back to `../data`). If it can't
find the data, upload the CSVs into the session or set `DATA_DIR` by hand and re-run.

In [ ]:
try:
    import google.colab            # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Reading uploaded files from the local session filesystem (not Google Drive).
MARKERS = ["edu_train.csv", "contrastive_dataset.csv"]
def _has_data(d):
    return bool(d) and any(os.path.exists(os.path.join(d, m)) for m in MARKERS)

def find_data_dir():
    env = os.environ.get("FALLACY_DATA_DIR")
    if _has_data(env):
        return env
    # common session locations (Colab's working dir is /content)
    for c in ["../data", "data", ".", "/content/data", "/content"]:
        if _has_data(c):
            return c
    # last resort: search the session filesystem (not Drive)
    for root in (["/content"] if IN_COLAB else ["."]):
        for m in MARKERS:
            hits = glob.glob(os.path.join(root, "**", m), recursive=True)
            if hits:
                return os.path.dirname(hits[0])
    return "../data"

DATA_DIR = find_data_dir()
# Save models into the session too. NOTE: session storage is ephemeral —
# download the saved models (or re-run) before the Colab session ends.
MODELS_ROOT = "/content/models" if IN_COLAB else "../models"

print("IN_COLAB:", IN_COLAB, "| DATA_DIR:", DATA_DIR, "| data found:", _has_data(DATA_DIR))
if not _has_data(DATA_DIR):
    print("  -> upload the CSVs (or a data/ folder) into the session, or set DATA_DIR manually")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - enable via Runtime > Change runtime type > GPU (or set RUN_BERT=False below)")

## 1. Config

In [ ]:
DATA_DIR = globals().get("DATA_DIR", "../data")
RUN_BERT = True         # set False to skip DistilBERT (CPU-only / quick smoke runs)

# Which model to use in the assembled pipeline for each stage
# ("bert" if trained, else falls back automatically)
DETECTOR_PREF = "bert"
TYPER_PREF    = "bert"

# Custom BiLSTM
MAX_LEN, MIN_FREQ = 80, 2
EMB_DIM, HIDDEN_DIM = 128, 128
BILSTM_EPOCHS, BILSTM_LR, BATCH_SIZE = 25, 1e-3, 32
DROPOUT, PATIENCE = 0.5, 4

# DistilBERT
BERT_MODEL, BERT_EPOCHS, BERT_LR, BERT_BATCH = "distilbert-base-uncased", 4, 2e-5, 16

## 2. Load all datasets
Everything is standardized to two columns: `text` and integer label `y`.
- **Detector** ← `contrastive_dataset.csv` (binary), split 70/15/15.
- **Typer** ← LOGIC `edu_train/dev/test.csv` (predefined splits, 13 classes).
- **Cross-domain test** ← `climate_test.csv` (same 13 classes, real-world).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_csv(name, text_col, label_col):
    df = pd.read_csv(os.path.join(DATA_DIR, name))[[text_col, label_col]].copy()
    df.columns = ["text", "label"]
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip()
    return df[df["text"].str.len() > 0].dropna().reset_index(drop=True)

# ---- Stage 1: detector data (binary) ----
det = load_csv("contrastive_dataset.csv", "text", "label")
det_le = LabelEncoder().fit(det["label"]); det["y"] = det_le.transform(det["label"])
DET_CLASSES = list(det_le.classes_)
det_tv, det_test = train_test_split(det, test_size=0.15, stratify=det["label"], random_state=SEED)
det_train, det_val = train_test_split(det_tv, test_size=0.15/0.85, stratify=det_tv["label"], random_state=SEED)
det_train, det_val, det_test = (d.reset_index(drop=True) for d in (det_train, det_val, det_test))

# ---- Stage 2: typer data (13-class, predefined splits) ----
typ_train = load_csv("edu_train.csv", "source_article", "updated_label")
typ_val   = load_csv("edu_dev.csv",   "source_article", "updated_label")
typ_test  = load_csv("edu_test.csv",  "source_article", "updated_label")
typ_le = LabelEncoder().fit(pd.concat([typ_train["label"], typ_val["label"], typ_test["label"]]))
TYP_CLASSES = list(typ_le.classes_)
for d in (typ_train, typ_val, typ_test):
    d["y"] = typ_le.transform(d["label"])

# ---- Cross-domain (OOD) test for the typer ----
climate = load_csv("climate_test.csv", "source_article", "logical_fallacies")
climate = climate[climate["label"].isin(set(TYP_CLASSES))].reset_index(drop=True)
climate["y"] = typ_le.transform(climate["label"])

print(f"Detector: train={len(det_train)} val={len(det_val)} test={len(det_test)}  classes={DET_CLASSES}")
print(f"Typer:    train={len(typ_train)} val={len(typ_val)} test={len(typ_test)}  classes={len(TYP_CLASSES)}")
print(f"OOD climate: {len(climate)}")

## 3. Quick EDA

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
det["label"].value_counts().plot(kind="bar", ax=ax[0], color="#2b7fb8", rot=0)
ax[0].set_title("Detector — fallacy vs valid (balanced)")
typ_all = pd.concat([typ_train, typ_val, typ_test])["label"].value_counts()
typ_all.plot(kind="barh", ax=ax[1], color="#5f8b4c"); ax[1].invert_yaxis()
ax[1].set_title("Typer — 13 fallacy types (imbalanced)")
plt.tight_layout(); plt.show()
print("typer class counts:\n", typ_all.to_string())

## 4. Shared evaluation helper
Every model, both stages, reports through this.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def evaluate(name, y_true, y_pred, class_names, show_report=False, plot_cm=False):
    acc = accuracy_score(y_true, y_pred)
    mf1 = f1_score(y_true, y_pred, average="macro")
    print(f"  {name:32s}  acc={acc:.3f}  macro-F1={mf1:.3f}")
    if show_report:
        print(classification_report(y_true, y_pred, labels=range(len(class_names)),
                                     target_names=class_names, zero_division=0))
    if plot_cm:
        cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
        plt.figure(figsize=(min(1+0.6*len(class_names), 12), min(1+0.5*len(class_names), 10)))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=class_names, yticklabels=class_names, cbar=False)
        plt.title(f"{name} — confusion matrix"); plt.ylabel("true"); plt.xlabel("pred")
        plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()
    return acc, mf1

## 5. Reusable models
Three trainers with a common interface: each takes dataframes with `text` + `y` and returns a
"bundle"; each has a matching `predict_*(bundle, df)`. Written once, used for **both** stages.

In [ ]:
# ---------- TF-IDF + Logistic Regression ----------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

def train_tfidf(train_df, val_df=None, num_classes=None, verbose=False):
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, stop_words="english")),
        ("clf", LogisticRegression(max_iter=2000, C=5.0, class_weight="balanced")),
    ])
    pipe.fit(train_df["text"], train_df["y"])
    return pipe

def predict_tfidf(bundle, df):
    return bundle.predict(df["text"])

In [ ]:
# ---------- Custom BiLSTM (from scratch, PyTorch) ----------
PAD, UNK = "<pad>", "<unk>"
def tokenize(t): return re.findall(r"[a-z0-9']+", t.lower())

def build_vocab(texts):
    c = Counter(tok for t in texts for tok in tokenize(t))
    itos = [PAD, UNK] + [w for w, n in c.most_common() if n >= MIN_FREQ]
    return itos, {w: i for i, w in enumerate(itos)}

def encode(text, stoi):
    ids = [stoi.get(tok, stoi[UNK]) for tok in tokenize(text)][:MAX_LEN]
    return ids or [stoi[UNK]]

class TextDS(Dataset):
    def __init__(self, df, stoi):
        self.x = [encode(t, stoi) for t in df["text"]]; self.y = df["y"].tolist()
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.y[i]

def make_collate(pad_idx):
    def collate(batch):
        seqs, ys = zip(*batch)
        lengths = torch.tensor([len(s) for s in seqs])
        padded = torch.full((len(seqs), lengths.max().item()), pad_idx, dtype=torch.long)
        for i, s in enumerate(seqs):
            padded[i, :len(s)] = torch.tensor(s, dtype=torch.long)
        return padded, lengths, torch.tensor(ys, dtype=torch.long)
    return collate

def make_loader(df, stoi, pad_idx, shuffle=False):
    return DataLoader(TextDS(df, stoi), batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=make_collate(pad_idx))

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, pad_idx):
        super().__init__()
        self.pad_idx = pad_idx
        self.emb = nn.Embedding(vocab_size, EMB_DIM, padding_idx=pad_idx)
        self.lstm = nn.LSTM(EMB_DIM, HIDDEN_DIM, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(HIDDEN_DIM * 2, num_classes)
    def forward(self, x, lengths):
        out, _ = self.lstm(self.emb(x))
        mask = (x != self.pad_idx).unsqueeze(-1)
        pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1)   # masked mean-pool
        return self.fc(self.dropout(pooled))

def train_bilstm(train_df, val_df, num_classes, verbose=True):
    itos, stoi = build_vocab(train_df["text"]); pad_idx = stoi[PAD]
    model = BiLSTMClassifier(len(itos), num_classes, pad_idx).to(DEVICE)
    freq = Counter(train_df["y"].tolist())
    w = torch.tensor([len(train_df) / (num_classes * freq[c]) for c in range(num_classes)],
                     dtype=torch.float).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam(model.parameters(), lr=BILSTM_LR, weight_decay=1e-4)
    tl = make_loader(train_df, stoi, pad_idx, shuffle=True)
    vl = make_loader(val_df, stoi, pad_idx)
    def run(loader, train):
        model.train() if train else model.eval(); P, G = [], []
        with torch.set_grad_enabled(train):
            for x, lengths, y in loader:
                x, y = x.to(DEVICE), y.to(DEVICE); logits = model(x, lengths)
                loss = crit(logits, y)
                if train:
                    opt.zero_grad(); loss.backward(); opt.step()
                P += logits.argmax(1).cpu().tolist(); G += y.cpu().tolist()
        return f1_score(G, P, average="macro")
    best, best_state, wait = -1, None, 0
    for ep in range(1, BILSTM_EPOCHS + 1):
        trf = run(tl, True); vaf = run(vl, False)
        if verbose: print(f"    epoch {ep:2d}  train_f1={trf:.3f}  val_f1={vaf:.3f}")
        if vaf > best:
            best, best_state, wait = vaf, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= PATIENCE: break
    if best_state: model.load_state_dict(best_state)
    return {"model": model, "stoi": stoi, "itos": itos, "pad_idx": pad_idx}

def predict_bilstm(bundle, df):
    m = bundle["model"]; m.eval(); P = []
    with torch.no_grad():
        for x, lengths, y in make_loader(df, bundle["stoi"], bundle["pad_idx"]):
            P += m(x.to(DEVICE), lengths).argmax(1).cpu().tolist()
    return P

In [ ]:
# ---------- DistilBERT (fine-tuned) ----------
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class BertDS(Dataset):
    def __init__(self, df, tokenizer):
        enc = tokenizer(list(df["text"]), truncation=True, max_length=MAX_LEN,
                        padding="max_length", return_tensors="pt")
        self.input_ids, self.attention_mask = enc["input_ids"], enc["attention_mask"]
        self.y = torch.tensor(df["y"].tolist(), dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return {"input_ids": self.input_ids[i], "attention_mask": self.attention_mask[i], "labels": self.y[i]}

def train_bert(train_df, val_df, num_classes, verbose=True):
    tok = AutoTokenizer.from_pretrained(BERT_MODEL)
    tl = DataLoader(BertDS(train_df, tok), batch_size=BERT_BATCH, shuffle=True)
    vl = DataLoader(BertDS(val_df, tok), batch_size=BERT_BATCH)
    m = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL, num_labels=num_classes).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=BERT_LR, weight_decay=0.01)
    def run(loader, train):
        m.train() if train else m.eval(); P, G = [], []
        with torch.set_grad_enabled(train):
            for b in loader:
                b = {k: v.to(DEVICE) for k, v in b.items()}; out = m(**b)
                if train:
                    opt.zero_grad(); out.loss.backward(); opt.step()
                P += out.logits.argmax(1).cpu().tolist(); G += b["labels"].cpu().tolist()
        return f1_score(G, P, average="macro")
    best, best_state, wait = -1, None, 0
    for ep in range(1, BERT_EPOCHS + 1):
        trf = run(tl, True); vaf = run(vl, False)
        if verbose: print(f"    epoch {ep}  train_f1={trf:.3f}  val_f1={vaf:.3f}")
        if vaf > best:
            best, best_state, wait = vaf, {k: v.cpu().clone() for k, v in m.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= 2: break
    if best_state: m.load_state_dict(best_state)
    return {"model": m, "tokenizer": tok}

def predict_bert(bundle, df):
    m = bundle["model"]; m.eval(); P = []
    with torch.no_grad():
        for b in DataLoader(BertDS(df, bundle["tokenizer"]), batch_size=BERT_BATCH):
            b = {k: v.to(DEVICE) for k, v in b.items()}
            P += m(**b).logits.argmax(1).cpu().tolist()
    return P

TRAIN   = {"tfidf": train_tfidf, "bilstm": train_bilstm, "bert": train_bert}
PREDICT = {"tfidf": predict_tfidf, "bilstm": predict_bilstm, "bert": predict_bert}
LABELS  = {"tfidf": "TF-IDF", "bilstm": "BiLSTM", "bert": "DistilBERT"}

In [ ]:
RESULTS, OOD_RESULTS = {}, {}

def run_stage(name, train_df, val_df, test_df, class_names, ood_df=None):
    """Train all models for one stage, evaluate on test (+ optional OOD), return bundles."""
    keys = ["tfidf", "bilstm"] + (["bert"] if RUN_BERT else [])
    bundles = {}
    print(f"\n===== {name}: training =====")
    for k in keys:
        print(f"  [{LABELS[k]}]")
        bundles[k] = TRAIN[k](train_df, val_df, len(class_names), verbose=(k != "tfidf"))
    print(f"\n===== {name}: test results =====")
    for k in keys:
        acc, mf1 = evaluate(f"{name}: {LABELS[k]}", test_df["y"].values, PREDICT[k](bundles[k], test_df), class_names)
        RESULTS[f"{name}: {LABELS[k]}"] = {"accuracy": acc, "macro_f1": mf1}
    if ood_df is not None:
        print(f"\n===== {name}: cross-domain (OOD) =====")
        for k in keys:
            acc, mf1 = evaluate(f"{name} OOD: {LABELS[k]}", ood_df["y"].values, PREDICT[k](bundles[k], ood_df), class_names)
            OOD_RESULTS[f"{name} OOD: {LABELS[k]}"] = {"accuracy": acc, "macro_f1": mf1}
    return bundles

## 6. Stage 1 — Detector (fallacy vs valid)
Trained on the balanced counterfactual set. Binary, so accuracy is meaningful here too.

In [ ]:
detector_bundles = run_stage("Detect", det_train, det_val, det_test, DET_CLASSES)

## 7. Stage 2 — Typer (13 fallacy types)
Trained on LOGIC `edu` only, then tested **cross-domain** on real-world climate claims.
Watch the in-domain → OOD macro-F1 drop.

In [ ]:
typer_bundles = run_stage("Type", typ_train, typ_val, typ_test, TYP_CLASSES, ood_df=climate)

### Confusion matrix — best typer (which fallacies get confused?)

In [ ]:
best_typer_key = "bert" if "bert" in typer_bundles else "bilstm"
_ = evaluate(f"Typer: {LABELS[best_typer_key]}", typ_test["y"].values,
             PREDICT[best_typer_key](typer_bundles[best_typer_key], typ_test),
             TYP_CLASSES, show_report=True, plot_cm=True)

## 8. Results summary

In [ ]:
summary = (pd.DataFrame(RESULTS).T.rename(columns={"accuracy": "Accuracy", "macro_f1": "Macro-F1"}))
print("In-domain test:")
display(summary.style.format("{:.3f}").background_gradient(cmap="Greens"))
if OOD_RESULTS:
    print("Cross-domain (typer on climate):")
    display(pd.DataFrame(OOD_RESULTS).T.rename(columns={"accuracy": "Accuracy", "macro_f1": "Macro-F1"})
            .style.format("{:.3f}"))

ax = summary.sort_values("Macro-F1")["Macro-F1"].plot(kind="barh", figsize=(8, 4.5), color="#2b7fb8")
ax.set_xlim(0, 1); ax.set_title("Macro-F1 by stage and model"); ax.set_xlabel("macro-F1")
for c in ax.containers: ax.bar_label(c, fmt="%.2f", padding=2, fontsize=8)
plt.tight_layout(); plt.show()

## 9. Assemble the pipeline
`detect → (if fallacy) type`. Uses your preferred model per stage (falls back if BERT wasn't run).

In [ ]:
DET_KEY = DETECTOR_PREF if DETECTOR_PREF in detector_bundles else "tfidf"
TYP_KEY = TYPER_PREF   if TYPER_PREF   in typer_bundles   else "tfidf"
print(f"pipeline: detector={LABELS[DET_KEY]}  typer={LABELS[TYP_KEY]}")

def classify(text):
    row = pd.DataFrame({"text": [str(text)], "y": [0]})
    verdict = det_le.inverse_transform(PREDICT[DET_KEY](detector_bundles[DET_KEY], row))[0]
    if verdict != "fallacy":
        return "valid — no fallacy detected"
    ftype = typ_le.inverse_transform(PREDICT[TYP_KEY](typer_bundles[TYP_KEY], row))[0]
    return f"fallacy → {ftype}"

demo = [
    "You're wrong about the budget because you never even finished college.",
    "Unemployment fell to 4% the year after the reform, which analysts partly credit to the new incentives.",
    "If we let students retake one exam, soon they'll expect to retake the entire year.",
    "Everyone I know is buying this stock, so it's obviously a safe bet.",
    "The bridge inspection found three cracked supports, so it should be closed until repaired.",
]
for s in demo:
    print(f"{s[:66]!r:70} => {classify(s)}")

## 10. Save the pipeline
Writes both stages to `MODELS_ROOT` (set in the Environment cell — the local session on Colab).
⚠️ **Colab session storage is ephemeral** — download the saved folder (Files pane → right-click →
Download) before the session ends. Reload these later to serve the model behind the app's old
interface — **zero API cost**.

In [ ]:
import joblib, json

save_dir = os.path.join(MODELS_ROOT, "two_stage"); os.makedirs(save_dir, exist_ok=True)

joblib.dump(det_le, os.path.join(save_dir, "detector_label_encoder.joblib"))
joblib.dump(typ_le, os.path.join(save_dir, "typer_label_encoder.joblib"))

def save_bundle(bundle, key, prefix):
    if key == "tfidf":
        joblib.dump(bundle, os.path.join(save_dir, f"{prefix}_tfidf.joblib"))
    elif key == "bilstm":
        torch.save({"state_dict": bundle["model"].state_dict(), "itos": bundle["itos"],
                    "pad_idx": bundle["pad_idx"], "num_classes": bundle["model"].fc.out_features,
                    "config": {"emb_dim": EMB_DIM, "hidden_dim": HIDDEN_DIM, "dropout": DROPOUT, "max_len": MAX_LEN}},
                   os.path.join(save_dir, f"{prefix}_bilstm.pt"))
    elif key == "bert":
        bundle["model"].save_pretrained(os.path.join(save_dir, f"{prefix}_distilbert"))
        bundle["tokenizer"].save_pretrained(os.path.join(save_dir, f"{prefix}_distilbert"))

save_bundle(detector_bundles[DET_KEY], DET_KEY, "detector")
save_bundle(typer_bundles[TYP_KEY], TYP_KEY, "typer")
json.dump({"detector_model": DET_KEY, "typer_model": TYP_KEY,
           "detector_classes": DET_CLASSES, "typer_classes": TYP_CLASSES, "max_len": MAX_LEN},
          open(os.path.join(save_dir, "pipeline.json"), "w"), indent=2)
print("saved to:", os.path.abspath(save_dir))
print("contents:", os.listdir(save_dir))

## 11. Takeaways & next steps

Fill in with your own numbers:

- **Detector** accuracy/F1 (this should be the easy stage) and **typer** macro-F1.
- **Generalization gap** — typer macro-F1 on LOGIC test vs. on climate (OOD). A big drop is the
  honest, interesting finding to discuss.
- **Hardest fallacy types** from the confusion matrix, and how the class imbalance
  (`faulty generalization` 319 vs `equivocation` 39) plays in.
- Whether **DistilBERT** justifies its cost over the from-scratch **BiLSTM** and the TF-IDF baseline.

Extensions: GloVe embeddings for the BiLSTM · add **MAFALDA** as a hard held-out benchmark ·
serve the saved pipeline behind the app (replacing the paid API).